# Anomaly Overlay Visualization
Lock in an anomaly detection method and target column in the first cell, then see where those anomalies occurred across different features.

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.getcwd())

from fetch_binance import fetch_klines
from anomaly_baseline import BaselineDetector
from signature_ocsvm import SignatureOCSVMDetector

# Fetch data
df = fetch_klines("BTCUSDT", "1m", "2023-10-01", "2023-10-03")
df.head()

,open,high,low,close,volume
open_time,,,,,
2023-10-01 00:00:00+00:00,26962.57,26962.57,26960.13,26960.13,3.11266
2023-10-01 00:01:00+00:00,26960.13,26960.14,26957.54,26957.54,3.78114
2023-10-01 00:02:00+00:00,26957.55,26957.55,26955.30,26955.31,5.50638
2023-10-01 00:03:00+00:00,26955.31,26955.31,26955.30,26955.31,5.64351
2023-10-01 00:04:00+00:00,26955.31,26965.88,26955.30,26965.87,8.96482


In [10]:
# =========================================================
# SELECTOR CELL
# =========================================================
# Options for DETECTOR_CLASS:
# - SignatureOCSVMDetector
# - BaselineDetector
#
# Options for TARGET_COL:
# - 'close' (Price)
# - 'volume'
# - 'taker_buy_base'
# - 'taker_sell_base'
# - 'imbalance'
# =========================================================

DETECTOR_CLASS = SignatureOCSVMDetector
TARGET_COL = "taker_buy_base"

print(f"Running {DETECTOR_CLASS.__name__} on column: '{TARGET_COL}'")
detector = DETECTOR_CLASS(target_col=TARGET_COL, window=60)
detector.fit(df)
anomalies = detector.predict(df)
print(f"Detected {len(anomalies)} anomalies.")

Running SignatureOCSVMDetector on column: 'taker_buy_base'
Extracting signatures for 'taker_buy_base' (level 2, window 60)...


KeyError: 'taker_buy_base'

In [ ]:
def plot_overlay(df, anomalies, plot_col, title):
    plt.figure(figsize=(15, 5))
    plt.plot(df.index, df[plot_col], label=f'BTCUSDT {plot_col.capitalize()}', color='black', linewidth=1, alpha=0.7)
    
    anomaly_values = df.loc[anomalies, plot_col]
    plt.scatter(anomaly_values.index, anomaly_values.values, color='red', label='Anomaly', zorder=5, s=20)
    
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Overlay on Price (Close)

In [ ]:
plot_overlay(df, anomalies, "close", f"Anomalies (detected on {TARGET_COL}) overlaid on PRICE")

### Overlay on Taker Buy Base (Aggressive Buying)

In [ ]:
plot_overlay(df, anomalies, "taker_buy_base", f"Anomalies (detected on {TARGET_COL}) overlaid on TAKER BUY BASE")